# 부서별 질문셋 RAG 임베딩 시스템

- **데이터**: `부서별_질문셋.xlsx` → `질문셋` 시트 (14개 부서, 질문 167개)
- **임베딩**: `BAAI/bge-m3` (dense 1024차원, 한국어 지원)
- **벡터DB**: ChromaDB (로컬 영구 저장, cosine 유사도, 부서 메타데이터 필터)

> 첫 실행 시 BGE-M3 모델(약 2.3GB)이 다운로드됩니다. GPU가 있으면 훨씬 빠릅니다.

## 0. 패키지 설치

In [42]:
import pandas, sentence_transformers, chromadb
print(pandas.__version__, sentence_transformers.__version__, chromadb.__version__)

2.2.3 5.7.0 1.5.9


## 1. 설정

In [43]:
from google.colab import drive
import os

# 1. 구글 드라이브 마운트 확인
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. 'Team 1 Project 3' 디렉토리 내 모든 파일 스캔 및 출력
target_dir = '/content/drive/MyDrive/Team 1 Project 3'

print(f"📂 '{target_dir}' 내부 파일 목록:")
if os.path.exists(target_dir):
    all_files = []
    for root, dirs, files in os.walk(target_dir):
        for file in files:
            relative_path = os.path.relpath(os.path.join(root, file), target_dir)
            all_files.append(relative_path)

    if all_files:
        for f in all_files:
            print(f" - {f}")
    else:
        print("   (폴더가 비어 있거나 하위 파일이 없습니다.)")
else:
    print("❌ 지정된 폴더가 드라이브에 존재하지 않습니다.")

📂 '/content/drive/MyDrive/Team 1 Project 3' 내부 파일 목록:
 - requirements.txt
 - 부서별_질문셋.xlsx.gsheet
 - rag.zip
 - Demo.ipynb
 - welfare_rag_baseline.ipynb
 - rag_embedding.ipynb
 - rag/.gitignore
 - rag/config.toml
 - rag/requirements.txt
 - rag/README.md
 - rag/rag/text.py
 - rag/rag/goldenset.py
 - rag/rag/config.py
 - rag/rag/splitter.py
 - rag/rag/bm25.py
 - rag/rag/store.py
 - rag/rag/embedder.py
 - rag/rag/pipeline.py
 - rag/rag/__init__.py
 - rag/rag/__main__.py
 - rag/rag/retriever.py
 - rag/rag/classifier.py
 - rag/rag/__pycache__/__init__.cpython-313.pyc
 - rag/rag/__pycache__/classifier.cpython-313.pyc
 - rag/rag/__pycache__/config.cpython-313.pyc
 - rag/rag/__pycache__/embedder.cpython-313.pyc
 - rag/rag/__pycache__/pipeline.cpython-313.pyc
 - rag/rag/__pycache__/goldenset.cpython-313.pyc
 - rag/rag/__pycache__/retriever.cpython-313.pyc
 - rag/rag/__pycache__/bm25.cpython-313.pyc
 - rag/rag/__pycache__/text.cpython-313.pyc
 - rag/rag/__pycache__/store.cpython-313.pyc

In [44]:
from pathlib import Path
import pandas as pd

XLSX_PATH  = "부서별_질문셋.xlsx"
SHEET      = "질문셋"
MODEL_NAME = "BAAI/bge-m3"
DB_DIR     = "./chroma_db"
COLLECTION = "dept_questions"
BATCH_SIZE = 32

REQUIRED_COLS = ["질문ID", "부서코드", "부서번호", "부서명", "교육 목적", "질문 No.", "질문 예시"]

## 2. 데이터 로드 & 문서화

질문 문장만 임베딩하면 "현재 담당하는 업무는?" 같은 질문이 부서 구분 없이 뭉치므로, **부서명·교육 목적을 함께 넣어** 문맥을 살립니다.

In [45]:
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_ID = "1HQZuYV6k4v3t0D2zIRy_xyd9oXK-5_6NkkRS7IS1peg"  # 부서별_질문셋.xlsx (Google Sheet)
ws = gc.open_by_key(SHEET_ID).worksheet(SHEET)
df = pd.DataFrame(ws.get_all_records())

missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing, f"필요한 컬럼이 없습니다: {missing}"

df["질문 예시"] = df["질문 예시"].astype(str).str.strip()
df = df[df["질문 예시"] != ""].copy()   # 빈 셀은 NaN이 아니라 ""로 들어옴
print(f"질문 수: {len(df)} / 부서 수: {df['부서코드'].nunique()}")
display(df.head())

질문 수: 167 / 부서 수: 14


,질문ID,부서코드,부서번호,부서명,교육 목적,질문 No.,질문 예시
0,EDU-01-Q01,EDU-01,1,경영기획·전략 부문,경영전략 및 조직 성과 향상,1,현재 담당하고 있는 경영기획 업무는 무엇인가요?
1,EDU-01-Q02,EDU-01,1,경영기획·전략 부문,경영전략 및 조직 성과 향상,2,중장기 경영전략 수립 경험이 있나요?
2,EDU-01-Q03,EDU-01,1,경영기획·전략 부문,경영전략 및 조직 성과 향상,3,공공기관 경영평가에 대한 이해도가 어느 정도인가요?
3,EDU-01-Q04,EDU-01,1,경영기획·전략 부문,경영전략 및 조직 성과 향상,4,KPI 및 성과지표를 설정할 수 있나요?
4,EDU-01-Q05,EDU-01,1,경영기획·전략 부문,경영전략 및 조직 성과 향상,5,경영성과 분석 및 보고서 작성 교육이 필요한가요?


In [46]:
  try:
      client.delete_collection(COLLECTION)
  except Exception:
      pass

In [47]:
def make_text(r):
    return f"부서: {r['부서명']}\n교육 목적: {r['교육 목적']}\n질문: {r['질문 예시']}"

ids   = df["질문ID"].astype(str).tolist()
texts = df.apply(make_text, axis=1).tolist()
metas = [
    {
        "dept_code": str(r["부서코드"]),
        "dept_no":   int(r["부서번호"]),
        "dept_name": str(r["부서명"]),
        "purpose":   str(r["교육 목적"]),
        "q_no":      int(r["질문 No."]),
        "question":  r["질문 예시"],
    }
    for _, r in df.iterrows()
]
print(texts[0])
print(metas[0])

부서: 경영기획·전략 부문
교육 목적: 경영전략 및 조직 성과 향상
질문: 현재 담당하고 있는 경영기획 업무는 무엇인가요?
{'dept_code': 'EDU-01', 'dept_no': 1, 'dept_name': '경영기획·전략 부문', 'purpose': '경영전략 및 조직 성과 향상', 'q_no': 1, 'question': '현재 담당하고 있는 경영기획 업무는 무엇인가요?'}


## 3. 임베딩 모델 로드 (BGE-M3)

In [48]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print("device:", device, "| dim:", model.get_sentence_embedding_dimension())

def embed(texts):
    # 정규화 → 내적 = 코사인 유사도
    return model.encode(
        texts, batch_size=BATCH_SIZE, normalize_embeddings=True,
        show_progress_bar=len(texts) > BATCH_SIZE,
    ).tolist()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

device: cpu | dim: 1024


/tmp/ipykernel_1514/1221757017.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("device:", device, "| dim:", model.get_sentence_embedding_dimension())


## 4. 임베딩 & ChromaDB 저장

In [49]:
import chromadb

client = chromadb.PersistentClient(path=DB_DIR)

# 다시 빌드할 때 기존 컬렉션 초기화
if COLLECTION in [c.name for c in client.list_collections()]:
    client.delete_collection(COLLECTION)
col = client.get_or_create_collection(name=COLLECTION, metadata={"hnsw:space": "cosine"})

vectors = embed(texts)
col.upsert(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
print(f"✅ {col.count()}개 질문 임베딩 완료 → {DB_DIR}")

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

✅ 167개 질문 임베딩 완료 → ./chroma_db


## 5. 검색 함수

In [50]:
def search(query, top_k=5, dept_code=None, min_score=None):
    """질의와 유사한 질문 검색. dept_code로 부서 필터 (예: 'EDU-11')."""
    res = col.query(
        query_embeddings=embed([query]),
        n_results=min(top_k, col.count()),
        where={"dept_code": dept_code} if dept_code else None,
    )
    hits = []
    for id_, meta, dist in zip(res["ids"][0], res["metadatas"][0], res["distances"][0]):
        score = 1.0 - dist  # cosine distance → similarity
        if min_score is not None and score < min_score:
            continue
        hits.append({"id": id_, "score": round(score, 4), **meta})
    return hits

def show(hits):
    return pd.DataFrame(hits)[["score", "id", "dept_name", "question"]]

In [51]:
show(search("신호 장애가 났을 때 대응할 수 있는지 묻는 질문", top_k=5))

,score,id,dept_name,question
0,0.6199,EDU-07-Q06,전기·통신·신호 부문,신호장치 장애 발생 시 원인을 분석할 수 있나요?
1,0.6115,EDU-05-Q05,운전·수송 부문,운행 중 신호 장애 발생 시 대응할 수 있나요?
2,0.5956,EDU-07-Q12,전기·통신·신호 부문,설비 장애 대응 능력을 실습으로 평가할 필요가 있나요?
3,0.5663,EDU-07-Q03,전기·통신·신호 부문,전철전력 및 전력공급 설비를 점검할 수 있나요?
4,0.5597,EDU-07-Q05,전기·통신·신호 부문,철도 통신망 및 무선통신 장비를 운영할 수 있나요?


In [52]:
# 부서 필터: IT·정보보안 부문만
show(search("보안 교육이 필요한가", top_k=3, dept_code="EDU-11"))

,score,id,dept_name,question
0,0.6639,EDU-11-Q07,IT·정보보안 부문,랜섬웨어 및 사이버 공격 대응 교육이 필요한가요?
1,0.6482,EDU-11-Q12,IT·정보보안 부문,업무 자동화 및 RPA 교육이 필요한가요?
2,0.6459,EDU-11-Q11,IT·정보보안 부문,시스템 접근권한 및 계정 관리 교육이 필요한가요?


## 6. LLM 프롬프트용 컨텍스트 만들기

In [53]:
def to_context(hits):
    return "\n".join(
        f"[{h['id']}] ({h['dept_name']} / 교육 목적: {h['purpose']}) {h['question']}"
        for h in hits
    )

query = "안전교육 이수 여부를 확인하는 질문이 뭐가 있어?"
context = to_context(search(query, top_k=5))

prompt = f"""다음은 부서별 교육 수요 조사 질문셋에서 검색된 질문들입니다.

{context}

사용자 질문: {query}
위 질문셋을 근거로 답변하세요."""
print(prompt)

다음은 부서별 교육 수요 조사 질문셋에서 검색된 질문들입니다.

[EDU-10-Q13] (안전·재난관리 부문 / 교육 목적: 철도 사고 예방 및 재난 대응 역량 강화) 부서별 안전교육 이수율을 관리하고 있나요?
[EDU-10-Q12] (안전·재난관리 부문 / 교육 목적: 철도 사고 예방 및 재난 대응 역량 강화) 안전교육 이수 후 현장 위험요인이 감소했는지 평가하나요?
[EDU-10-Q07] (안전·재난관리 부문 / 교육 목적: 철도 사고 예방 및 재난 대응 역량 강화) 심폐소생술 및 응급처치 교육을 이수했나요?
[EDU-10-Q03] (안전·재난관리 부문 / 교육 목적: 철도 사고 예방 및 재난 대응 역량 강화) 산업안전보건 교육을 정기적으로 이수하고 있나요?
[EDU-08-Q06] (토목·시설 부문 / 교육 목적: 철도 선로 및 기반시설 유지보수 능력 향상) 선로 작업 중 안전수칙을 숙지하고 있나요?

사용자 질문: 안전교육 이수 여부를 확인하는 질문이 뭐가 있어?
위 질문셋을 근거로 답변하세요.


## 7. 저장된 인덱스 다시 불러오기 (다음 세션)

모델 로드(3번 셀) 후 아래만 실행하면 재임베딩 없이 바로 검색할 수 있습니다.

In [54]:
client = chromadb.PersistentClient(path=DB_DIR)
col = client.get_collection(COLLECTION)
print("저장된 문서 수:", col.count())

저장된 문서 수: 167
